# AgriSwin-MoE: train on Colab

Runs the **same** `train.py`, recipe and frozen splits as the laptop, on a Colab
L4 or A100. Nothing about training lives in this notebook: it prepares the
runtime and starts the queue runner, so a Colab result can share a table with a
laptop result.

**Before you run anything**

1. **Runtime > Change runtime type > L4 GPU** (or A100 GPU) > Save.
2. *Recommended:* add a `GITHUB_TOKEN` secret (the key icon in the left sidebar)
   so each result is pushed to GitHub the moment its run finishes. Without it,
   results are kept on your Google Drive. How to make the token:
   `docs/NEXT_STEPS.md`, Part C.
3. **Runtime > Run all**, and keep this tab open while it trains.

**If the runtime disconnects**, just *Run all* again: finished runs are skipped
and an interrupted one resumes from its last finished epoch, because
checkpoints live on your Drive.

In [1]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
NVIDIA L4, 23034 MiB, 580.82.07


In [2]:
# 2. Connect Google Drive. Checkpoints and the image cache live there, so they
#    survive a disconnect. The first time, Colab asks you to allow access.
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
# 3. Get the latest code
import os
REPO = "/content/agriswin-moe"
if os.path.isdir(f"{REPO}/.git"):
    !git -C {REPO} pull --ff-only
else:
    !git clone https://github.com/amritaj312-max/agriswin-moe.git {REPO}
%cd {REPO}
!git log --oneline -1

Cloning into '/content/agriswin-moe'...
remote: Enumerating objects: 325, done.
remote: Counting objects: 100% (325/325), done.
remote: Compressing objects: 100% (165/165), done.
remote: Total 325 (delta 167), reused 300 (delta 145), pack-reused 0 (from 0)
Receiving objects: 100% (325/325), 4.86 MiB | 23.35 MiB/s, done.
Resolving deltas: 100% (167/167), done.
/content/agriswin-moe
e7de484 (HEAD -> main, origin/main, origin/HEAD) Result: swinv2_tiny_seed1 -- test acc 99.80%, macro-F1 99.66%, ECE 0.15%


In [4]:
# 4. Prepare this runtime: exact packages, images verified byte for byte
#    against the laptop's, checkpoints on Drive. The first time this downloads
#    the images (a few minutes) and caches them on Drive; after that it copies
#    the cache (about a minute).
import importlib
import sys
sys.path.insert(0, f"{REPO}/notebooks")
import colab_runtime
importlib.reload(colab_runtime)
runtime = colab_runtime.setup()

GPU NVIDIA L4 (23034 MiB), 12 CPUs, 53 GB RAM -> 8 data-loading workers

Copying images from /content/drive/MyDrive/agriswin/data/agriswin_images.tar ...

Checking every image against the reference the results were made on...
PlantVillage:
  train   37992 images hashed in 5s
  val      8158 images hashed in 1s
  test     8155 images hashed in 1s
  train  identical to the reference
  val    identical to the reference
  test   identical to the reference

PlantDoc evaluation set:
  all 2551 images present by content, fingerprint 2f8dd8c1e3e71d08

OK: every image is byte-for-byte the one the results were made on.

  code         e7de484 Result: swinv2_tiny_seed1 -- test acc 99.80%, macro-F1 99.66%, ECE 0.15%
  packages     timm==1.0.30
  images       verified, from the cache on Drive
  checkpoints  /content/drive/MyDrive/agriswin/checkpoints
  results      committed and pushed to GitHub as each run finishes


## What to train

A queue file holds one `train.py` argument line per run. `colab_first.txt` is
the first job: EfficientNet-B0 seed 42 again, which must land inside the
laptop's spread (proving the GPU does not move results), then Swin-V2-T seeds 1
and 2, which give Phase 3's ablation control a seed floor of its own.

**Do not change the batch size or learning rate** for the bigger GPU: every row
of a table must share one recipe, and `train.py` refuses a paper run that
changes it. The speed-up comes from the GPU alone.

In [5]:
# 5. Choose the queue
QUEUE = "ml_pipeline/configs/queues/colab_first.txt"  #@param {type:"string"}
!cat {QUEUE}

# The first Colab queue. One train.py argument line per run; finished runs are
# skipped and an interrupted one resumes, so after a disconnect run it again.
#   notebooks/train_on_colab.ipynb runs this file.
#
# 1. EfficientNet-B0, seed 42: the shortest run, so it proves the whole Colab
#    setup end to end before the long ones. It is the same configuration as
#    results/efficientnet_b0.json, but on another machine "seed 42" is in
#    effect a new random draw, so it cannot show the two machines equivalent
#    (it scored 99.67% against the laptop's 99.75-99.79%; see NEXT_STEPS H3).
#    That is why each comparison table is trained on a single machine.
# 2-3. Swin-V2-T, seeds 1 and 2. Phase 3's ablations are compared against
#    plain Swin-V2-T, and a difference only counts if it clears that model's
#    own seed floor -- which needs more than the one seed trained so far.
--config ml_pipeline/configs/baselines/efficientnet_b0.yaml --set run_name=efficientnet_b0_colab
--config ml_pi

In [6]:
# 6. Train. Each result is committed, and pushed with the token, as soon as
#    its run finishes. The img/s figure in the progress lines is this GPU's
#    real speed on this recipe.
NUM_WORKERS = runtime["num_workers"]
TRAILER = f"Trained on Colab ({runtime['gpu']})"
!python ml_pipeline/src/training/run_queue.py {QUEUE} --set-all num_workers={NUM_WORKERS} --commit-trailer "{TRAILER}"

[queue 2026-11-02 12:30:01] ml_pipeline/configs/queues/colab_first.txt: 3 run(s), pid 1760
[queue 2026-11-02 12:30:01] efficientnet_b0_colab: already finished -- skipped
[queue 2026-11-02 12:30:01] swinv2_tiny_seed1: already finished -- skipped
[queue 2026-11-02 12:30:01] swinv2_tiny_seed2: resuming
run     swinv2_tiny_seed2  (resuming after epoch 14)
model   swinv2_tiny_window8_256  27.61 M params  weights timm/swinv2_tiny_window8_256.ms_in1k
data    train 37,992 / val 8,158 / test 8,155
input   256px  mean (0.485, 0.456, 0.406)  std (0.229, 0.224, 0.225)
recipe  AdamW lr 0.0001 wd 0.05  warmup 2 ep, cosine to 1e-06  40 ep x 1187 steps  batch 32  float16
  epoch 15 [ 100/1187]  loss 0.0358  lr 7.73e-05  134 img/s  eta 4m19s
  epoch 15 [ 200/1187]  loss 0.0355  lr 7.70e-05  152 img/s  eta 3m27s
  epoch 15 [ 300/1187]  loss 0.0334  lr 7.67e-05  152 img/s  eta 3m07s
  epoch 15 [ 400/1187]  loss 0.0330  lr 7.64e-05  151 img/s  eta 2m46s
  epoch 15 [ 500/1187]  loss 0.0329  lr 7.61e-05  15

In [7]:
# 7. Every result so far, from every machine. The Splits and Recipe lines at
#    the bottom must say all runs match.
!python ml_pipeline/src/evaluation/summarize_results.py

run                         params   epochs  test acc  macro-F1     ECE     NLL    GPU   notes
----------------------------------------------------------------------------------------------
efficientnet_b0              4.06M   34/40     99.75%    99.64%   0.17%  0.0090   2.1h   7 epoch(s) stalled, 2.6h wall clock
efficientnet_b0_colab        4.06M   17/27     99.67%    99.50%   0.21%  0.0112    45m   early stop
efficientnet_b0_seed1        4.06M   27/37     99.77%    99.65%   0.13%  0.0073   1.9h   early stop, 5 epoch(s) stalled, 3.0h wall clock
efficientnet_b0_seed2        4.06M   32/40     99.79%    99.69%   0.14%  0.0090   2.1h   
mobilevit_s                  4.96M   40/40     99.79%    99.65%   0.17%  0.0085   4.1h   1 epoch(s) stalled, 4.4h wall clock
swinv2_tiny                 27.61M   28/38     99.82%    99.78%   0.13%  0.0074   7.3h   early stop, 1 epoch(s) stalled, 8.3h wall clock
swinv2_tiny_seed1           27.61M   35/40     99.80%    99.66%   0.15%  0.0095   3.0h   
swinv2

In [8]:
!cd /content/agriswin-moe && git pull --rebase && git push

Already up to date.
Everything up-to-date
